# Day 5b — Timed simulation (45 minutes)
### Ebury technical interview prep (Treasury/ALM)

**Before you start:**
1. Set a timer for 45 minutes.
2. Don't look at the Day 1-4 notebooks while doing this -- if you get stuck, move on or leave a note and come back later.
3. Don't look at the solution (clearly separated at the end) until you finish or time runs out.
4. Talk through what you're doing out loud, as if you were in the real interview -- that's part of the practice.

**Files needed in the same folder:** `simulation_transactions.csv` and `simulation_counterparties.csv`.

---
## Task

You work on the Treasury team. You've been given two files:
- `simulation_transactions.csv`: FX transactions with currency, amount, counterparty and maturity date.
- `simulation_counterparties.csv`: reference table with each counterparty's name and type.

**You are asked to deliver:**

1. **Clean data**: no duplicates, currency normalized (uppercase), and the maturity date correctly parsed (formats come mixed).
2. **Enriched dataset**: transactions joined with counterparty information (name and client type).
3. **Liquidity gap ladder**: net exposure accumulated by currency, in tenor buckets: `0-7d`, `7-30d`, `30-90d`, `90-180d`, `>180d` (today = August 31, 2026).
4. **Alerts**: flag any currency/bucket combination where the cumulative gap is negative and exceeds, in absolute value, 150,000.
5. **A simple chart** of the gap ladder.
6. **Bonus, if you have time left**: replicate the total amount by `client_type` (Corporate vs SME) using SQL (DuckDB) instead of pandas.

No guided `# TODO` cells this time -- the cell below is your free workspace. Add as many cells as you need.

In [ ]:
# Your workspace -- add as many cells as you need
import pandas as pd
import numpy as np
import duckdb








---
---
# ⏱️ DO NOT KEEP READING UNTIL YOU FINISH OR TIME RUNS OUT
---
---




## Full step-by-step solution

### Step 1 — Load and explore (assume nothing)

In [ ]:
df = pd.read_csv("simulation_transactions.csv")
counterparties = pd.read_csv("simulation_counterparties.csv")

print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print("Duplicates:", df.duplicated().sum())
df.head()


### Step 2 — Cleaning

In [ ]:
df_clean = df.copy()
df_clean = df_clean.drop_duplicates()
df_clean["Ccy"] = df_clean["Ccy"].str.upper()
df_clean["Maturity"] = pd.to_datetime(df_clean["Maturity"], format="mixed", errors="coerce")

print(df_clean.dtypes)
print(df_clean.isna().sum())


### Step 3 — Enrich with counterparties

In [ ]:
df_enriched = df_clean.merge(counterparties, on="CpID", how="left")
df_enriched.head()


### Step 4 — Gap ladder

In [ ]:
today = pd.Timestamp("2026-08-31")
df_enriched["days"] = (df_enriched["Maturity"] - today).dt.days

bins = [0, 7, 30, 90, 180, np.inf]
labels = ["0-7d", "7-30d", "30-90d", "90-180d", ">180d"]
df_enriched["tenor_bucket"] = pd.cut(df_enriched["days"], bins=bins, labels=labels, ordered=True)

table = df_enriched.dropna(subset=["Amount"]).pivot_table(
    index="tenor_bucket", columns="Ccy", values="Amount",
    aggfunc="sum", fill_value=0, observed=False
)
table = table.reindex(labels)
gap_ladder = table.cumsum()
print(gap_ladder)


### Step 5 — Alerts

In [ ]:
for bucket in gap_ladder.index:
    for ccy in gap_ladder.columns:
        value = gap_ladder.loc[bucket, ccy]
        if value < -150000:
            print(f"Alert: {ccy} in {bucket} -> cumulative gap {value:,.0f}")


### Step 6 — Chart

In [ ]:
import matplotlib.pyplot as plt

gap_ladder.plot(kind="bar", figsize=(8, 4))
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Cumulative liquidity gap ladder")
plt.tight_layout()
plt.show()


### Bonus — the same breakdown by client_type, in SQL

In [ ]:
query = """
SELECT ClientType, SUM(Amount) as total
FROM df_enriched
WHERE Amount IS NOT NULL
GROUP BY ClientType
"""
print(duckdb.sql(query).df())


---
## Self-assessment

Compare your result against the solution on these specific points, not just "did it run":

- Did you spot all 3 data issues yourself (duplicate row, mixed-case currency, mixed date formats) before the solution pointed them out, or did you miss some?
- Did you use `ordered=True` in `pd.cut()` without being reminded?
- Is your gap ladder an actual **cumulative** total (cumsum on the already-aggregated table), or did you only compute the net per bucket without accumulating? That's the easiest conceptual mistake to make under time pressure.
- Did you make it to the SQL bonus in time, or did the pandas part eat up all your time?

If you ran out of time, identify exactly which step slowed you down the most -- that's the most useful signal for what to review in your remaining days.